### 1. 들어가며

In [1]:
!pip install datasets
!pip install loralib
!pip install trl==0.11.0
!pip install transformers==4.40.0
!pip install accelerate


In [ ]:
# !git clone https://github.com/airobotlab/KoChatGPT
# !cp -r KoChatGPT/colossalai_ChatGPT_230319/chatgpt chatgpt

fatal: destination path 'KoChatGPT' already exists and is not an empty directory.


In [3]:
# !rm -rf chatgpt

In [2]:
import os

modifications = [
    {
        "file": "chatgpt/trainer/callbacks/save_checkpoint.py",
        "changes": [
            {"line": 3, "old": "from chatgpt.trainer.strategies import ColossalAIStrategy, Strategy",
             "new": "from chatgpt.trainer.strategies import Strategy"},
            {"line": 71, "old": "only_rank0 = not isinstance(self.strategy, ColossalAIStrategy)",
             "new": "            only_rank0 = not isinstance(self.strategy)"},
        ],
    },
    {
        "file": "chatgpt/trainer/strategies/__init__.py",
        "changes": [
            {"line": 1, "old": "from .colossalai import ColossalAIStrategy", "new": ""},  # 삭제
            {"line": 5, "old": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy', 'ColossalAIStrategy']",
             "new": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy']"},
        ],
    },
    {
        "file": "chatgpt/dataset/reward_dataset.py",
        "changes": [
            {"line": 3, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ],
    },
    {
        "file": "chatgpt/trainer/strategies/__init__.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    },
    {
        "file": "chatgpt/dataset/reward_dataset.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    }
]


def modify_file(file_path, changes):
    """파일에서 지정된 줄을 찾아 내용을 수정하는 함수"""

    if not os.path.exists(file_path):
        print(f"⚠️ 파일이 존재하지 않습니다: {file_path}")
        return

    with open(file_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    modified = False

    for change in changes:
        line_index = change["line"]
        if 0 <= line_index < len(lines):
            if lines[line_index].strip() == change["old"]:
                lines[line_index] = change["new"] + "\n"
                modified = True
            else:
                print(f"⚠️ {file_path} 파일의 {change['line']}번째 줄이 예상과 다릅니다.")
                print(f"   예상: {change['old']}")
                print(f"   실제: {lines[line_index].strip()}")

    if modified:
        with open(file_path, "w", encoding="utf-8") as file:
            file.writelines(lines)
        print(f"✅ 수정 완료: {file_path}")
    else:
        print(f"⚠️ {file_path} 수정할 내용이 없습니다.")

for mod in modifications:
    modify_file(mod["file"], mod["changes"])

⚠️ chatgpt/trainer/callbacks/save_checkpoint.py 파일의 3번째 줄이 예상과 다릅니다.
   예상: from chatgpt.trainer.strategies import ColossalAIStrategy, Strategy
   실제: from chatgpt.trainer.strategies import Strategy
⚠️ chatgpt/trainer/callbacks/save_checkpoint.py 파일의 71번째 줄이 예상과 다릅니다.
   예상: only_rank0 = not isinstance(self.strategy, ColossalAIStrategy)
   실제: only_rank0 = not isinstance(self.strategy)
⚠️ chatgpt/trainer/callbacks/save_checkpoint.py 수정할 내용이 없습니다.
⚠️ chatgpt/trainer/strategies/__init__.py 파일의 1번째 줄이 예상과 다릅니다.
   예상: from .colossalai import ColossalAIStrategy
   실제: 
⚠️ chatgpt/trainer/strategies/__init__.py 파일의 5번째 줄이 예상과 다릅니다.
   예상: __all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy', 'ColossalAIStrategy']
   실제: __all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy']
⚠️ chatgpt/trainer/strategies/__init__.py 수정할 내용이 없습니다.
⚠️ chatgpt/dataset/reward_dataset.py 파일의 3번째 줄이 예상과 다릅니다.
   예상: from tqdm import tqdm
   실제: from tqdm.notebook import tqdm
⚠️ chatgpt/dataset/reward_dataset.py 수

In [5]:
import os
import json
from dataclasses import dataclass
from typing import Dict, List, Any

import torch
from torch.utils.data import Dataset
from datasets import load_dataset

from transformers import ElectraTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from trl import PPOTrainer, PPOConfig

from torch.utils.data import DataLoader
from torch.optim import AdamW
from math import ceil

from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead

from tqdm import tqdm


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [6]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

if device == "cuda":
    torch.cuda.empty_cache()
    print(torch.cuda.get_device_name(0))


NVIDIA GeForce RTX 4080 Laptop GPU


In [35]:
# ============================
# 0. CONFIGS
# ============================

@dataclass
class DataConfig:
    sft_path: str = "KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl"
    rm_path: str  = "KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl"
    ppo_path: str = "KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl"
    max_length_sft: int = 512
    max_length_ppo_query: int = 128
    max_length_ppo_response: int = 128

@dataclass
class ModelConfig:
    koelectra_name: str = "monologg/koelectra-base-v3-discriminator"
    base_lm_name: str   = "monologg/koelectra-base-v3-generator"
    sft_actor_ckpt: str = "checkpoints/sft_actor"
    ppo_actor_ckpt: str = "checkpoints/ppo_actor"
    rm_ckpt: str        = "checkpoints/rm_model"

@dataclass
class TrainConfig:
    # SFT
    sft_batch_size: int = 8
    sft_epochs: int = 1
    sft_max_steps: int = 1000
    sft_lr: float = 5e-5
    run_sft_if_ckpt_exists: bool = True  # 체크포인트 있어도 다시 학습할지 여부

    # RM
    rm_batch_size: int = 8
    rm_epochs: int = 1
    rm_max_steps: int = 1000
    rm_lr: float = 5e-5
    run_rm_if_ckpt_exists: bool = True

    # PPO
    ppo_batch_size: int = 2
    ppo_mini_batch_size: int = 2
    ppo_epochs: int = 3
    ppo_total_steps: int = 200
    ppo_learning_rate: float = 5e-6
    run_ppo_if_ckpt_exists: bool = True

    # PPO KL 관련
    ppo_kl_target: float = 0.03
    ppo_kl_init_coef: float = 0.03
    ppo_cliprange: float = 0.15
    ppo_cliprange_value: float = 0.15
    ppo_gamma: float = 0.99
    ppo_lam: float = 0.95


@dataclass
class GenerationConfig:
    max_new_tokens: int = 128
    do_sample: bool = True
    top_k: int = 50
    top_p: float = 0.95
    temperature: float = 1.0
    eos_token_id: int = None      # 나중에 tokenizer.eos_token_id로 채움
    pad_token_id: int = None

data_cfg = DataConfig()
model_cfg = ModelConfig()
train_cfg = TrainConfig()
gen_cfg = GenerationConfig()


### 2. Base model and Dataset for RLHF

In [32]:
# 0-1 KoELECTRA tokenizer
tokenizer = ElectraTokenizer.from_pretrained(model_cfg.koelectra_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # decoder-only 안정성

gen_cfg.eos_token_id = tokenizer.eos_token_id
gen_cfg.pad_token_id = tokenizer.pad_token_id

# causal LM 에 맞게 eos/pad 토큰 설정 (필요시 조정)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# 0-2 Base LM KoGPT2 for SFT
actor_model = AutoModelForCausalLM.from_pretrained(model_cfg.base_lm_name).to(device)
actor_model.resize_token_embeddings(len(tokenizer))

config.json:   0%|          | 0.00/463 [00:00<?, ?B/s]

c:\Users\itmin\vscode-workspace\sample_python\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\itmin\.cache\huggingface\hub\models--monologg--koelectra-base-v3-generator. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/149M [00:00<?, ?B/s]

If you want to use `ElectraForCausalLM` as a standalone, add `is_decoder=True.`


Embedding(35000, 768, padding_idx=0)

In [33]:
tokenizer.model_max_length

512

### 데이터셋 확인

In [10]:
with open(data_cfg.sft_path, "r", encoding='utf-8') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

12000


[{'prompt': '불고기용 고기 한우에요?',
  'completion': "'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.",
  'tokens': 193},
 {'prompt': '쓰던 앱이 유료로 전환됐어',
  'completion': "'어떤 앱인지 모르기 때문에 정확한 답변을 드리기 어렵습니다. 하지만, 일반적으로 유료 전환된 앱은 기존 무료 앱에서 추가적인 기능이나 서비스를 제공하는 경우가 많습니다. 이 경우, 유료 전환 이전 가격이 매우 저렴하거나 무료인 경우가 많으므로, 앱을 계속 사용하려면 추가적인 비용을 지불해야 할 수 있습니다. 하지만, 유료 전환 이후에는 기존 사용자 또는 새로운 사용자 모두가 추가 기능과 높은 품질의 서비스를 이용할 수 있게 됩니다. 따라서, 앱 개발자는 유료 모델로 전환함으로써 앱의 수익을 증가시키고 더 나은 서비스를 제공할 수 있습니다.",
  'tokens': 288},
 {'prompt': '여친이랑 다툼',
  'completion': "'저는 인공지능이기 때문에 여친과의 다툼에 대해서는 조언을 드릴 수는 없습니다. 하지만 다투게 되면 상대방의 말을 경청하는 태도와 서로의 감정을 존중하는 것이 중요합니다. 논리적이고 차분한 대화를 통해 문제를 해결해보세요. 그리고 서로를 이해하고 서로의 의견을 수용하는 것이 중요합니다.",
  'tokens': 153}]

In [11]:
with open(data_cfg.rm_path, "r", encoding='utf-8') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

10220


[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?',
  'completion_0': 'Allow me to answer your question. I know that you are curious about me.',
  'completion_1': '번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.',
  'completion_2': '라이언에게 말했다.',
  'ranking': [2, 1, 0]},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?',
  'completion_0': '개포주공아파트는 다섯 단지로 이루어져 있습니다.',
  'completion_1': '이날 목송에서 구글상위노',
  'completion_2': '개포주공아파트는 총 27개 단지로 이루어져 있습니다.',
  'ranking': [2, 0, 1]},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?',
  'completion_0': 'The diameter of the Metallic domain is bigger than the Hyperonic domain.',
  'completion_1': '이 질문은 조금 불분명합니다. 김영삼 대통령이 후보 시절에 어떤 발언을 했고, 누가 그 발언을 문제삼았는지에 따라 답이 다를 수 있습니다.\\n\\n만약 김영삼 대통령이 후보 시절에 지역표심을 겨냥한 발언을 했다는 가정하에, 그 발언을 문제삼은 후보가 누구였는지를 대답하자면, 그 답은 이화선 당시 민주당 대통령 후보가 될 것입니다. 1992년 총선 때, 김영삼 대선후보는 "집값이 오른 노량진역 부근의 부동산 가격은 세월호 폭침 후 \\\'강남 도시재생\\\' 일환으로 상승했다"는 발언을 했습니다. 하지만 이화선 후보는 이 발언을 "전국적으로 경제적 발전이 이루어지지 않은 지방민의 마음을 멀리해지려는 무례한 발언"이라고 비판하며 문

In [12]:
with open(data_cfg.ppo_path, "r", encoding='utf-8') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

12000


[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?'},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?'},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?'}]

### 3. Supervised Fine-Tuning

In [36]:
# ===================================
# 1. SFT(Supervised Fine-tuning)용 데이터셋
#    (KoChatGPT SFT jsonl 사용)
# ===================================

class SFTDataset(Dataset):
    def __init__(self, path: str, max_length: int = 512):
        self.samples = []
        
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
        for obj in data:
            q = obj.get("prompt", "")
            a = obj.get("completion", "")
            if not q or not a:
                continue
            self.samples.append({"question": q, "answer": a})
            
        if not self.samples:
            raise ValueError(f"No valid samples found in path: {path}")
            
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        prompt = f"{sample['question']}"
        text = prompt + sample["answer"]
        
        enc = tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        labels = input_ids.clone()
        
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

sft_dataset = SFTDataset(data_cfg.sft_path, max_length=data_cfg.max_length_sft)
print(len(sft_dataset))


11997


In [37]:
# ===================================
# 2. SFT 학습 루프
# ===================================



def collate_fn_sft(batch):
    input_ids = torch.stack([b["input_ids"] for b in batch])
    attention_mask = torch.stack([b["attention_mask"] for b in batch])
    labels = torch.stack([b["labels"] for b in batch])
    return {
        "input_ids": input_ids.to(device),
        "attention_mask": attention_mask.to(device),
        "labels": labels.to(device),
    }

sft_loader = DataLoader(
    sft_dataset,
    batch_size=train_cfg.sft_batch_size,
    shuffle=True,
    collate_fn=collate_fn_sft,
)


sft_ckpt_exists = os.path.isdir(model_cfg.sft_actor_ckpt) and os.path.isfile(
    os.path.join(model_cfg.sft_actor_ckpt, "config.json")
)

if sft_ckpt_exists and not train_cfg.run_sft_if_ckpt_exists:
    print(f"[SFT] 체크포인트가 이미 존재하여 학습을 건너뜁니다: {model_cfg.sft_actor_ckpt}")
    # 필요하면 여기서 바로 로드만 수행
    actor_model = AutoModelForCausalLM.from_pretrained(model_cfg.sft_actor_ckpt).to(device)

else:
    print(f"[SFT] 학습을 수행합니다. (기존 ckpt 존재 여부: {sft_ckpt_exists})")

    optimizer = AdamW(actor_model.parameters(), lr=train_cfg.sft_lr)

    batch_size = sft_loader.batch_size
    total_steps = ceil(len(sft_dataset) / batch_size)
    max_steps = min(total_steps * train_cfg.sft_epochs, train_cfg.sft_max_steps)

    print("total_steps", total_steps, "max_steps", max_steps)

    sft_train_logs = [] 
    actor_model.train()
    step = 0
    for epoch in range(train_cfg.sft_epochs):
        for batch in sft_loader:
            if step >= max_steps:
                print(f"stop at step {step}/{total_steps}")
                break
            outputs = actor_model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            sft_train_logs.append({"epoch": epoch, "step": step, "loss": loss.item()})
    
            if step % 100 == 0:
                print(f"SFT epoch {epoch} step {step} loss {loss.item():.4f}")
            step += 1
        else:
            continue
        break
    
    os.makedirs(model_cfg.sft_actor_ckpt, exist_ok=True)
    actor_model.save_pretrained(model_cfg.sft_actor_ckpt)
    tokenizer.save_pretrained(model_cfg.sft_actor_ckpt)

[SFT] 학습을 수행합니다. (기존 ckpt 존재 여부: True)
total_steps 1500 max_steps 1000
SFT epoch 0 step 0 loss 10.8142
SFT epoch 0 step 100 loss 0.9207
SFT epoch 0 step 200 loss 0.2327
SFT epoch 0 step 300 loss 0.1137
SFT epoch 0 step 400 loss 0.1097
SFT epoch 0 step 500 loss 0.1210
SFT epoch 0 step 600 loss 0.0546
SFT epoch 0 step 700 loss 0.0585
SFT epoch 0 step 800 loss 0.0491
SFT epoch 0 step 900 loss 0.0512
stop at step 1000/1500


### 4. Reward Model

In [38]:
# ===================================
# 3. Reward Model (RM) - KoELECTRA 토크나이저 사용
#    KoChatGPT step2 jsonl 사용
# ===================================

class RMDataset(Dataset):
    def __init__(self, path: str, max_length: int = 512):
        self.samples = []
        self.max_length = max_length
        with open(path, "r", encoding="utf-8-sig") as f:
            data = json.load(f)

        print("RM raw items", len(data))
        for obj in data:
            prompt = obj.get("prompt", None)
            if not prompt:
                continue

            completions = [
                obj.get("completion_0", None),  # _ 추가
                obj.get("completion_1", None),  # _ 추가  
                obj.get("completion_2", None),  # _ 추가
            ]
            ranking = obj.get("ranking", None) or [2, 1, 0]

            for idx, comp in enumerate(completions):
                if comp is None:
                    continue
                text = f"{prompt} {comp}"
                try:
                    pos = ranking.index(idx)
                except ValueError:
                    pos = len(ranking) - 1
                if pos == 0:
                    reward = 1.0
                elif pos == 1:
                    reward = 0.5
                else:
                    reward = 0.0
                self.samples.append({"text": text, "reward": float(reward)})

        print("RM parsed samples", len(self.samples))
        if not self.samples:
            raise ValueError(f"No valid samples found in path: {path}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        enc = tokenizer(
            s["text"],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        reward = torch.tensor(s["reward"], dtype=torch.float)
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": reward,
        }

rm_dataset = RMDataset(data_cfg.rm_path, max_length=data_cfg.max_length_sft)
print("len(rm_dataset)", len(rm_dataset))

RM raw items 10220
RM parsed samples 30651
len(rm_dataset) 30651


In [39]:
def collate_fn_rm(batch):
    input_ids = torch.stack([b["input_ids"] for b in batch])
    attention_mask = torch.stack([b["attention_mask"] for b in batch])
    labels = torch.stack([b["labels"] for b in batch])
    return {
        "input_ids": input_ids.to(device),
        "attention_mask": attention_mask.to(device),
        "labels": labels.to(device),
    }

rm_loader = DataLoader(
    rm_dataset,
    batch_size=train_cfg.sft_batch_size,  # 혹은 별도 rm_batch_size 추가
    shuffle=True,
    collate_fn=collate_fn_rm,
)


rm_ckpt_exists = os.path.isdir(model_cfg.rm_ckpt) and os.path.isfile(
    os.path.join(model_cfg.rm_ckpt, "config.json")
)

if rm_ckpt_exists and not train_cfg.run_rm_if_ckpt_exists:
    print(f"[RM] 체크포인트가 이미 존재하여 학습을 건너뜁니다: {model_cfg.rm_ckpt}")
    reward_model = AutoModelForSequenceClassification.from_pretrained(
        model_cfg.rm_ckpt,
        num_labels=1,
    ).to(device)
else:
    print(f"[RM] 학습을 수행합니다. (기존 ckpt 존재 여부: {rm_ckpt_exists})")

    reward_model = AutoModelForSequenceClassification.from_pretrained(
        model_cfg.koelectra_name,
        num_labels=1,
    ).to(device)
    
    optimizer_rm = AdamW(reward_model.parameters(), lr=train_cfg.sft_lr)
    
    batch_size = rm_loader.batch_size
    steps_per_epoch = ceil(len(rm_dataset) / batch_size)
    max_steps = min(steps_per_epoch * train_cfg.sft_epochs, train_cfg.sft_max_steps)
    print("RM steps_per_epoch", steps_per_epoch, "max_steps", max_steps)
    
    reward_model.train()
    step = 0
    for epoch in range(train_cfg.sft_epochs):
        for batch in rm_loader:
            if step >= max_steps:
                print(f"RM stop at step {step}/{steps_per_epoch}")
                break
            outputs = reward_model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"].unsqueeze(-1),
            )
            loss = outputs.loss
            loss.backward()
            optimizer_rm.step()
            optimizer_rm.zero_grad()
    
            if step % 100 == 0:
                print(f"RM epoch {epoch} step {step} loss {loss.item():.4f}")
            step += 1
        else:
            continue
        break
    
    os.makedirs(model_cfg.rm_ckpt, exist_ok=True)
    reward_model.save_pretrained(model_cfg.rm_ckpt)
    tokenizer.save_pretrained(model_cfg.rm_ckpt)
    print(f"[RM] 체크포인트 저장 완료: {model_cfg.rm_ckpt}")

[RM] 학습을 수행합니다. (기존 ckpt 존재 여부: True)


c:\Users\itmin\vscode-workspace\sample_python\.venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RM steps_per_epoch 3832 max_steps 1000
RM epoch 0 step 0 loss 0.2495
RM epoch 0 step 100 loss 0.0345
RM epoch 0 step 200 loss 0.0331
RM epoch 0 step 300 loss 0.0454
RM epoch 0 step 400 loss 0.0574
RM epoch 0 step 500 loss 0.0067
RM epoch 0 step 600 loss 0.0182
RM epoch 0 step 700 loss 0.0355
RM epoch 0 step 800 loss 0.0092
RM epoch 0 step 900 loss 0.0096
RM stop at step 1000/3832
[RM] 체크포인트 저장 완료: checkpoints/rm_model


### 5. Proximal Policy Optimization

In [ ]:
# ===================================
# 4. PPO(강화학습) 단계 - TRL 사용
# ===================================

class PromptDataset(Dataset):
    def __init__(self, path: str):
        with open(path, "r", encoding="utf-8-sig") as f:
            data = json.load(f)

        self.prompts = [obj.get("prompt", None) for obj in data if obj.get("prompt", None)]
        print("PPO prompts:", len(self.prompts))

        if not self.prompts:
            raise ValueError(f"No valid prompts found in {path}")

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        return {"prompt": self.prompts[idx]}


ppo_dataset = PromptDataset(data_cfg.ppo_path)
print(len(ppo_dataset))


# ---------------------------
# tokenizer 안정화
# ---------------------------

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


ppo_ckpt_exists = os.path.isdir(model_cfg.ppo_actor_ckpt) and os.path.isfile(
    os.path.join(model_cfg.ppo_actor_ckpt, "config.json")
)

if ppo_ckpt_exists and not train_cfg.run_ppo_if_ckpt_exists:

    print(f"[PPO] 체크포인트가 이미 존재하여 학습을 건너뜁니다: {model_cfg.ppo_actor_ckpt}")

    # ❶ 저장된 PPO actor는 일반 CausalLM으로 로드한 뒤 ValueHead로 래핑
    base_actor = AutoModelForCausalLM.from_pretrained(model_cfg.ppo_actor_ckpt)
    actor_model = AutoModelForCausalLMWithValueHead(base_actor)

    actor_model.pretrained_model.resize_token_embeddings(len(tokenizer))
    actor_model.to(device)

else:

    print(f"[PPO] 학습을 수행합니다. (기존 ckpt 존재 여부: {ppo_ckpt_exists})")

    # ---------------------------
    # 1. SFT 모델 로드
    # ---------------------------

    sft_model = AutoModelForCausalLM.from_pretrained(model_cfg.sft_actor_ckpt)
    sft_model.resize_token_embeddings(len(tokenizer))
    sft_model.to(device)

    # ---------------------------
    # 2. PPO actor / ref
    # ---------------------------

    # ❷ SFT actor ckpt도 먼저 CausalLM으로 로드 후 래핑
    base_actor = AutoModelForCausalLM.from_pretrained(model_cfg.sft_actor_ckpt)
    base_actor.resize_token_embeddings(len(tokenizer))
    base_actor.to(device)

    ppo_actor = AutoModelForCausalLMWithValueHead(base_actor)
    ppo_actor.to(device)

    # ref 모델도 동일하게
    base_ref = AutoModelForCausalLM.from_pretrained(model_cfg.sft_actor_ckpt)
    base_ref.resize_token_embeddings(len(tokenizer))
    base_ref.to(device)

    ppo_ref = AutoModelForCausalLMWithValueHead(base_ref)
    ppo_ref.to(device)

    for p in ppo_ref.parameters():
        p.requires_grad = False

    # ---------------------------
    # 3. PPO Config
    # ---------------------------

    ppo_config = PPOConfig(
        batch_size=train_cfg.ppo_batch_size,
        mini_batch_size=train_cfg.ppo_mini_batch_size,
        learning_rate=train_cfg.ppo_learning_rate,
        log_with=None,
        optimize_cuda_cache=True,
        target_kl=train_cfg.ppo_kl_target,
        kl_penalty="kl",
        init_kl_coef=train_cfg.ppo_kl_init_coef,
        ppo_epochs=train_cfg.ppo_epochs,
        cliprange=train_cfg.ppo_cliprange,
        cliprange_value=train_cfg.ppo_cliprange_value,
        gamma=train_cfg.ppo_gamma,
        lam=train_cfg.ppo_lam,
    )

    # ---------------------------
    # 4. PPO Trainer
    # ---------------------------

    ppo_trainer = PPOTrainer(
        config=ppo_config,
        model=ppo_actor,
        ref_model=ppo_ref,
        tokenizer=tokenizer,
        dataset=ppo_dataset,
    )

    # ---------------------------
    # 5. Reward
    # ---------------------------

    def compute_reward(prompts, responses):

        texts = [f"{p} {r}" for p, r in zip(prompts, responses)]

        enc = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=data_cfg.max_length_sft,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            out = reward_model(**enc)
            rewards = out.logits.squeeze(-1)

        return rewards

    # ---------------------------
    # 6. PPO Training Loop
    # ---------------------------

    ppo_trainer.model.train()

    total_steps = train_cfg.ppo_total_steps
    step = 0

    for epoch in range(1000000):
        for batch in tqdm(ppo_trainer.dataloader):

            if step >= total_steps:
                break

            prompts = batch["prompt"]

            enc = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=data_cfg.max_length_ppo_query,
            ).to(device)

            # tokenizer 기준 안전 token
            pad_token_id = tokenizer.pad_token_id
            eos_token_id = tokenizer.eos_token_id

            with torch.no_grad():

                gen_outputs = ppo_trainer.model.generate(
                    **enc,
                    max_new_tokens=gen_cfg.max_new_tokens,
                    pad_token_id=pad_token_id,
                    eos_token_id=eos_token_id,
                    do_sample=True,
                    top_k=50,
                    top_p=0.9,
                    temperature=0.7,
                )

            responses = []
            query_tensors = []
            response_tensors = []

            for i in range(len(prompts)):

                input_len = (enc["input_ids"][i] != pad_token_id).sum()

                out_ids = gen_outputs[i, input_len:]

                responses.append(
                    tokenizer.decode(out_ids, skip_special_tokens=True)
                )

                query_tensors.append(enc["input_ids"][i])
                response_tensors.append(out_ids)

            rewards = compute_reward(prompts, responses)

            # PPO는 list[tensor] 필요
            scores = [torch.tensor(r).to(device) for r in rewards]

            stats = ppo_trainer.step(
                queries=query_tensors,
                responses=response_tensors,
                scores=scores,
            )

            reward_mean = torch.stack(scores).mean().item()

            print(
                f"PPO step {step} "
                f"reward mean {reward_mean:.4f}, "
                f"loss_total {stats['ppo/loss/total']:.4f}"
            )

            step += 1

        if step >= total_steps:
            break

    # ---------------------------
    # 7. Save
    # ---------------------------

    os.makedirs(model_cfg.ppo_actor_ckpt, exist_ok=True)

    # ValueHead 래퍼 안의 pretrained_model 기준으로 저장됨
    ppo_trainer.model.save_pretrained(model_cfg.ppo_actor_ckpt)
    tokenizer.save_pretrained(model_cfg.ppo_actor_ckpt)

    print(f"[PPO] 체크포인트 저장 완료: {model_cfg.ppo_actor_ckpt}")

    actor_model = ppo_trainer.model


In [ ]:
#
# 1. 모델 로드
#
# 1) Base (pretrained)
base_model = AutoModelForCausalLM.from_pretrained(
    model_cfg.base_lm_name
).to(device)
base_model.eval()

# 2) SFT
sft_model = AutoModelForCausalLM.from_pretrained(
    model_cfg.sft_actor_ckpt
).to(device)
sft_model.eval()

# 3) PPO
ppo_model = AutoModelForCausalLM.from_pretrained(
    model_cfg.ppo_actor_ckpt
).to(device)
ppo_model.eval()


#
# 2. 공통 생성 함수
#
def generate_answer(model, prompt, max_new_tokens=gen_cfg.max_new_tokens):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=data_cfg.max_length_ppo_query,
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=gen_cfg.do_sample,
            top_k=gen_cfg.top_k,
            top_p=gen_cfg.top_p,
            temperature=gen_cfg.temperature,
            pad_token_id=gen_cfg.pad_token_id,
            eos_token_id=gen_cfg.eos_token_id,
        )

    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return text


#
# 3. Reward Model 점수 함수
# 
def score_with_rm(prompt, answer):
    reward_model.eval()
    text = f"{prompt} {answer}"
    enc = tokenizer(
        text,
        truncation=True,
        max_length=data_cfg.max_length_sft,
        padding="max_length",
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        out = reward_model(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
        )
    score = out.logits.squeeze(-1).item()
    return score


c:\Users\itmin\vscode-workspace\sample_python\.venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at checkpoints/ppo_actor were not used when initializing GPT2LMHeadModel: ['v_head.summary.bias', 'v_head.summary.weight']
- This IS expected if you are initializing GPT2LMHeadModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing GPT2LMHeadModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [30]:
#
# 4. 평가용 프롬프트 설정
# 
test_prompts = [
    "쓰던 앱이 유료로 전환됐어",
    "면접에서 자기소개를 잘하는 방법 알려줘",
    "기분이 우울할 때 어떻게 해야 해?",
]

# 
# 5. Base / SFT / PPO 비교 + RM 점수
# 
results = []

for p in test_prompts:
    print("=" * 80)
    print(f"[PROMPT]\n{p}\n")

    base_ans = generate_answer(base_model, p)
    sft_ans = generate_answer(sft_model, p)
    ppo_ans = generate_answer(ppo_model, p)

    base_score = score_with_rm(p, base_ans)
    sft_score = score_with_rm(p, sft_ans)
    ppo_score = score_with_rm(p, ppo_ans)

    results.append({
        "prompt": p,
        "base_answer": base_ans,
        "sft_answer": sft_ans,
        "ppo_answer": ppo_ans,
        "base_score": base_score,
        "sft_score": sft_score,
        "ppo_score": ppo_score,
    })

    print("[BASE]")
    print(base_ans, "\n")
    print(f"RM score: {base_score:.3f}\n")

    print("[SFT]")
    print(sft_ans, "\n")
    print(f"RM score: {sft_score:.3f}\n")

    print("[PPO]")
    print(ppo_ans, "\n")
    print(f"RM score: {ppo_score:.3f}\n")

# 전체 평균 점수
import numpy as np

base_mean = np.mean([r["base_score"] for r in results])
sft_mean = np.mean([r["sft_score"] for r in results])
ppo_mean = np.mean([r["ppo_score"] for r in results])

print("=" * 80)
print("[RM score mean]")
print(f"Base: {base_mean:.3f}")
print(f"SFT : {sft_mean:.3f}")
print(f"PPO : {ppo_mean:.3f}")


[PROMPT]
쓰던 앱이 유료로 전환됐어

[BASE]
쓰던 앱이 유료로 전환됐어 快 罪 껍 껍 껍 껍 綠 껍 껍 綠 껍 綠 껍 綠 綠 껍 綠 綠 껍 綠 껍 綠 綠 綠 綠 껍 껍 綠 綠 껍 껍 綠 綠 綠 綠 껍 綠 綠 껍 綠 綠 껍 껍 綠 껍 껍 綠 綠 껍 껍 껍 綠 껍 綠 綠 綠 껍 綠 껍 껍 綠 껍 껍 綠 綠 綠 綠 綠 껍 綠 綠 綠 껍 綠 綠 綠 綠 綠 껍 綠 껍 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 껍 껍 綠 綠 껍 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 껍 綠 綠 껍 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 

RM score: -0.025

[SFT]
쓰던 앱이 유료로 전환됐어 

RM score: 0.000

[PPO]
쓰던 앱이 유료로 전환됐어 

RM score: 0.000

[PROMPT]
면접에서 자기소개를 잘하는 방법 알려줘

[BASE]
면접에서 자기소개를 잘하는 방법 알려줘 규모 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 채 

RM score: -0.058

[SFT]
면접에서 자기소개를 잘하는 방법 알려줘 

RM score: 0.055

[PPO]
면접에서 자기소개를 잘하는 방법 알려줘 

RM score: 0.055

[PROMPT]
기분이 우울할 때 어떻게 해야 해?

[BASE]
기분이 우울할 때 어떻게 해야 해? 닻 랐 第 넜 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 껍 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 綠 